# 第七章：图驱动 Agent 与生产实践

## 学习目标

- 将图查询封装为 Agent 工具
- 构建能自主决策的知识图谱 Agent
- 使用 LangGraph 构建图推理工作流
- 掌握知识图谱的增量更新和维护
- 了解生产环境的性能优化策略

## 0. 环境准备

加载环境变量，初始化 LLM 和 Neo4jGraph，并写入科技公司示例数据。

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

# 切换为 GLM（Anthropic 兼容接口）：
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="glm-4-plus", base_url=os.environ["GLM_API_BASE"], api_key=os.environ["GLM_API_KEY"])

print("✓ LLM 初始化完成")

In [ ]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)
print("\u2713 Neo4jGraph 初始化成功")

In [ ]:
# 清空数据库，从零开始
graph.query("MATCH (n) DETACH DELETE n")
print("\u2713 数据库已清空")

In [ ]:
# 写入科技公司知识图谱（与前面章节的数据集一致）
graph.query("""
    CREATE (bj:City {name: '北京'})
    CREATE (sh:City {name: '上海'})
    CREATE (sz:City {name: '深圳'})
    CREATE (hz:City {name: '杭州'})

    CREATE (hw:Company {name: '华为', founded: 1987, industry: '通信技术'})
    CREATE (tx:Company {name: '腾讯', founded: 1998, industry: '互联网'})
    CREATE (ali:Company {name: '阿里巴巴', founded: 1999, industry: '电子商务'})
    CREATE (bd:Company {name: '百度', founded: 2000, industry: '搜索引擎'})
    CREATE (jd:Company {name: '京东', founded: 1998, industry: '电子商务'})
    CREATE (byte:Company {name: '字节跳动', founded: 2012, industry: '互联网'})

    CREATE (rrz:Person {name: '任正非'})
    CREATE (mht:Person {name: '马化腾'})
    CREATE (jym:Person {name: '马云'})
    CREATE (lyh:Person {name: '李彦宏'})
    CREATE (lqd:Person {name: '刘强东'})
    CREATE (zym:Person {name: '张一鸣'})

    CREATE (rrz)-[:FOUNDED]->(hw)
    CREATE (mht)-[:FOUNDED]->(tx)
    CREATE (jym)-[:FOUNDED]->(ali)
    CREATE (lyh)-[:FOUNDED]->(bd)
    CREATE (lqd)-[:FOUNDED]->(jd)
    CREATE (zym)-[:FOUNDED]->(byte)

    CREATE (hw)-[:LOCATED_IN]->(sz)
    CREATE (tx)-[:LOCATED_IN]->(sz)
    CREATE (ali)-[:LOCATED_IN]->(hz)
    CREATE (bd)-[:LOCATED_IN]->(bj)
    CREATE (jd)-[:LOCATED_IN]->(bj)
    CREATE (byte)-[:LOCATED_IN]->(bj)

    CREATE (wx:Product {name: '微信', category: '社交'})
    CREATE (tb:Product {name: '淘宝', category: '电商'})
    CREATE (dy:Product {name: '抖音', category: '短视频'})
    CREATE (bds:Product {name: '百度搜索', category: '搜索'})
    CREATE (jdm:Product {name: '京东商城', category: '电商'})
    CREATE (qq:Product {name: 'QQ', category: '社交'})
    CREATE (zfb:Product {name: '支付宝', category: '支付'})
    CREATE (mate:Product {name: 'Mate 系列', category: '手机'})

    CREATE (tx)-[:HAS_PRODUCT]->(wx)
    CREATE (tx)-[:HAS_PRODUCT]->(qq)
    CREATE (ali)-[:HAS_PRODUCT]->(tb)
    CREATE (ali)-[:HAS_PRODUCT]->(zfb)
    CREATE (byte)-[:HAS_PRODUCT]->(dy)
    CREATE (bd)-[:HAS_PRODUCT]->(bds)
    CREATE (jd)-[:HAS_PRODUCT]->(jdm)
    CREATE (hw)-[:HAS_PRODUCT]->(mate)

    CREATE (tx)-[:COMPETES_WITH]->(byte)
    CREATE (ali)-[:COMPETES_WITH]->(jd)
    CREATE (tx)-[:COOPERATES_WITH]->(jd)
""")
print("\u2713 科技公司知识图谱创建完成")

In [ ]:
# 验证数据
result = graph.query("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY count DESC
""")
for r in result:
    print(f"  {r['label']}: {r['count']} 个节点")

# 刷新 Schema（写入数据后必须做）
graph.refresh_schema()
print(f"\nSchema 概要:\n{graph.schema[:300]}...")

### 刚才发生了什么？

我们创建了一个科技公司知识图谱，包含四类节点和五种关系：

| 节点标签 | 数量 | 示例 |
|---------|------|------|
| `Company` | 6 | 华为、腾讯、阿里巴巴 |
| `Person` | 6 | 任正非、马化腾、马云 |
| `City` | 4 | 北京、深圳、杭州 |
| `Product` | 8 | 微信、淘宝、抖音 |

| 关系类型 | 含义 | 示例 |
|---------|------|------|
| `FOUNDED` | 创立 | 马化腾 → 腾讯 |
| `LOCATED_IN` | 总部所在 | 腾讯 → 深圳 |
| `HAS_PRODUCT` | 拥有产品 | 腾讯 → 微信 |
| `COMPETES_WITH` | 竞争关系 | 腾讯 ↔ 字节跳动 |
| `COOPERATES_WITH` | 合作关系 | 腾讯 ↔ 京东 |

`graph.refresh_schema()` 让 Neo4jGraph 重新读取图结构——写入新数据后如果不刷新，`graph.schema` 返回的还是旧信息。

---

## 1. 图查询作为 Agent 工具

在 LangChain 第六章中，我们学过用 `@tool` 把普通函数变成 Agent 工具。这里我们做同样的事——只不过工具背后操作的是知识图谱。

In [ ]:
from langchain_core.tools import tool

@tool
def query_graph(cypher: str) -> str:
    """执行 Cypher 查询知识图谱。输入 Cypher 查询语句，返回结果。"""
    try:
        result = graph.query(cypher)
        return str(result[:10])  # 限制结果长度
    except Exception as e:
        return f"查询错误: {e}"

@tool
def search_entity(entity_name: str) -> str:
    """搜索实体的相关信息。输入实体名称，返回该实体的所有关系。"""
    result = graph.query(
        "MATCH (n)-[r]-(m) WHERE n.name = $name OR n.id = $name "
        "RETURN n.name AS source, type(r) AS relation, m.name AS target LIMIT 10",
        params={"name": entity_name},
    )
    if not result:
        return f"未找到实体: {entity_name}"
    lines = [f"{r['source']} -[{r['relation']}]-> {r['target']}" for r in result]
    return "\n".join(lines)

@tool
def get_graph_schema() -> str:
    """获取知识图谱的结构信息（节点类型、关系类型）。"""
    return graph.schema

print("已定义工具：")
for t in [query_graph, search_entity, get_graph_schema]:
    print(f"  {t.name}: {t.description[:40]}...")

### 刚才发生了什么？

我们定义了三个图查询工具，它们的分工各不相同：

| 工具 | 输入 | 用途 |
|------|------|------|
| `query_graph` | Cypher 语句 | 精确查询，Agent 需要自己写 Cypher |
| `search_entity` | 实体名称 | 探索性查询，快速获取某个实体的所有关系 |
| `get_graph_schema` | 无 | 获取图结构，帮助 Agent 理解数据模型 |

**和 LangChain 第六章的对比**：LangChain 第六章中 `@tool` 装饰的是计算器、天气查询等通用工具。这里的工具专门服务于知识图谱——本质上是把图数据库的能力暴露给 LLM。

> 注意 `query_graph` 的 `result[:10]` 限制：Agent 工具的返回值会作为 LLM 输入的一部分，过长的结果会消耗 token 并降低推理质量。生产环境中要根据 LLM 的上下文窗口大小来调整这个限制。

---

## 2. 构建知识图谱 Agent

在 LangGraph 第四章中，我们学过 `create_react_agent` 一行代码创建 ReAct Agent。现在把图查询工具交给它。

In [ ]:
from langgraph.prebuilt import create_react_agent

# 创建 ReAct Agent
tools = [query_graph, search_entity, get_graph_schema]
agent = create_react_agent(llm, tools)

# 测试
response = agent.invoke({
    "messages": [{"role": "user", "content": "腾讯和华为有什么关系？它们分别在哪个城市？"}]
})

# 打印最终回答
print(response["messages"][-1].content)

In [ ]:
# 查看 Agent 的推理过程
for msg in response["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{role}] 调用工具: {tc['name']}({tc['args']})")
    elif hasattr(msg, "content") and msg.content:
        print(f"[{role}] {str(msg.content)[:100]}...")

### 刚才发生了什么？

Agent 使用 ReAct 模式（Think → Act → Observe → Repeat）自主完成了查询：

```
用户提问 → Agent 思考 → 调用 search_entity("腾讯") → 获得腾讯的关系
         → Agent 思考 → 调用 search_entity("华为") → 获得华为的关系
         → Agent 综合两次结果 → 生成最终回答
```

Agent 可能会：
- 先调用 `get_graph_schema` 理解图结构
- 再调用 `search_entity` 探索实体关系
- 或直接调用 `query_graph` 执行精确的 Cypher 查询

**具体的调用顺序由 LLM 自主决定**——这正是 Agent 和固定 Chain 的核心区别。每次运行的路径可能不同，但都能得到正确答案。

> 回忆 LangGraph 第四章：`create_react_agent` 内部就是一个 StateGraph，包含 `agent` 节点（LLM 决策）和 `tools` 节点（工具执行），通过条件边连接。

---

## 3. 多步图推理

简单问题一步就能答，但有些问题需要多步推理——Agent 要先查一个信息，再基于结果查下一个。

In [ ]:
# 需要多步推理的复杂问题
complex_questions = [
    "找出在深圳的公司的创始人，以及这些创始人创立公司的年份",
    "哪些公司有社交类产品？这些公司分别在哪个城市？",
    "阿里巴巴和腾讯的竞争产品有哪些？",
]

for q in complex_questions:
    print(f"\nQ: {q}")
    response = agent.invoke({"messages": [{"role": "user", "content": q}]})
    print(f"A: {response['messages'][-1].content}")
    print("-" * 50)

### 刚才发生了什么？

这些问题的共同特点是：**答案不在一个查询里**。

以「找出在深圳的公司的创始人」为例，Agent 的推理链可能是：

1. 先查深圳有哪些公司：`MATCH (c:Company)-[:LOCATED_IN]->(city:City {name:'深圳'}) RETURN c.name`
2. 再查每个公司的创始人：`MATCH (p:Person)-[:FOUNDED]->(c:Company {name:'华为'}) RETURN p.name`
3. 综合结果给出完整回答

这就是 Agent 比 GraphCypherQAChain（第四章）更强大的地方——它不是一次性生成一条 Cypher，而是可以**根据中间结果动态调整后续查询**。

| 方式 | 特点 | 适合场景 |
|------|------|----------|
| GraphCypherQAChain | 一次生成一条 Cypher | 单步查询 |
| ReAct Agent | 多步推理，动态决策 | 复杂问题、需要组合多个查询 |

---

## 4. LangGraph 工作流：智能路由

在 LangGraph 第三章中我们学过条件边（conditional edges）。这里用同样的思路，根据问题类型选择不同的检索策略。

设计思路：

```
用户提问 → 分类（graph / vector / direct）
              ↓             ↓           ↓
          图检索       向量检索      直接回答
              ↓             ↓           ↓
              └──────── 生成回答 ────────┘
```

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class QueryState(TypedDict):
    question: str
    query_type: str   # "graph" | "vector" | "direct"
    context: str
    answer: str

In [ ]:
def classify_question(state: QueryState) -> dict:
    """分析问题类型"""
    prompt = f"""分析以下问题应该使用哪种方式回答：
- graph: 涉及实体关系、结构化查询（如"谁创立了X"、"X和Y的关系"）
- vector: 涉及语义理解、概述类问题（如"介绍一下X"）
- direct: 简单问题，不需要检索

问题：{state['question']}
只输出一个词：graph、vector 或 direct"""

    result = llm.invoke(prompt)
    query_type = result.content.strip().lower()
    if query_type not in ["graph", "vector", "direct"]:
        query_type = "graph"  # 默认走图检索
    return {"query_type": query_type}

In [ ]:
def graph_retrieval(state: QueryState) -> dict:
    """图检索：提取实体，查询知识图谱"""
    # 用 LLM 从问题中提取实体名称
    entity_result = llm.invoke(
        f"从以下问题中提取实体名称（人名、公司名、城市名等），用逗号分隔，只输出名称：{state['question']}"
    )
    entities = [e.strip() for e in entity_result.content.split(",")]

    context_parts = []
    for entity in entities:
        result = graph.query(
            "MATCH (n)-[r]-(m) WHERE n.name = $name "
            "RETURN n.name AS s, type(r) AS r, m.name AS t LIMIT 5",
            params={"name": entity},
        )
        context_parts.extend([f"{r['s']} -[{r['r']}]-> {r['t']}" for r in result])

    return {"context": "\n".join(context_parts) if context_parts else "未找到相关图数据"}

In [ ]:
def direct_answer(state: QueryState) -> dict:
    """直接回答：不需要检索，LLM 直接回答"""
    return {"context": "无需检索，直接回答"}

In [ ]:
def generate_answer(state: QueryState) -> dict:
    """基于上下文生成最终回答"""
    prompt = (
        f"基于以下信息回答问题。如果信息不足，请如实说明。\n\n"
        f"信息：{state['context']}\n\n"
        f"问题：{state['question']}\n"
        f"回答："
    )
    result = llm.invoke(prompt)
    return {"answer": result.content}

In [ ]:
def route_query(state: QueryState) -> Literal["graph_retrieval", "direct_answer"]:
    """路由函数：根据问题类型分发"""
    if state["query_type"] == "graph":
        return "graph_retrieval"
    else:
        return "direct_answer"

In [ ]:
# 构建工作流
workflow = StateGraph(QueryState)

# 添加节点
workflow.add_node("classify", classify_question)
workflow.add_node("graph_retrieval", graph_retrieval)
workflow.add_node("direct_answer", direct_answer)
workflow.add_node("generate_answer", generate_answer)

# 添加边
workflow.add_edge(START, "classify")
workflow.add_conditional_edges("classify", route_query)
workflow.add_edge("graph_retrieval", "generate_answer")
workflow.add_edge("direct_answer", "generate_answer")
workflow.add_edge("generate_answer", END)

app = workflow.compile()
print("\u2713 工作流编译完成")

In [ ]:
# 测试工作流
questions = [
    "腾讯是谁创立的？",
    "你好，今天天气怎么样？",
    "华为和哪些公司在同一城市？",
]

for q in questions:
    result = app.invoke({"question": q, "query_type": "", "context": "", "answer": ""})
    print(f"Q: {q}")
    print(f"路由: {result['query_type']}")
    print(f"A: {result['answer'][:100]}...")
    print()

### 刚才发生了什么？

我们用 LangGraph 构建了一个**智能路由工作流**：

1. **classify 节点**：用 LLM 判断问题类型（graph / vector / direct）
2. **条件边**：根据分类结果路由到不同的检索节点
3. **graph_retrieval 节点**：提取实体 → 查询知识图谱 → 拼接上下文
4. **direct_answer 节点**：不做检索，直接传递
5. **generate_answer 节点**：基于上下文生成最终回答

**和第 2 节 ReAct Agent 的对比**：

| | ReAct Agent | LangGraph 工作流 |
|---|---|---|
| **决策者** | LLM 全权决定工具调用顺序 | 开发者预定义流程，LLM 只在节点内工作 |
| **灵活性** | 高——可以自由组合工具 | 中——走预定义的分支 |
| **可控性** | 低——LLM 可能走错路 | 高——每一步都在预期内 |
| **适合场景** | 探索性问题、开放域 | 生产环境、流程明确的场景 |

**生产建议**：对于已知的查询模式，优先用 LangGraph 工作流（可控）；对于开放域问题，用 ReAct Agent（灵活）。

> 回忆 LangGraph 第三章的条件边语法：`add_conditional_edges(source_node, routing_function)` 中，路由函数的返回值必须是下游节点的名称。

---

## 5. 增量更新与图维护

知识图谱不是一次性构建的——新信息不断产生，图需要持续更新。第五章学过用 `LLMGraphTransformer` 从文本提取图数据，这里演示**增量写入**。

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document

# 新增文档
new_text = "小米由雷军于2010年在北京创立。小米的主要产品包括小米手机和米家智能家居。"
new_doc = [Document(page_content=new_text)]

# 提取图数据
transformer = LLMGraphTransformer(llm=llm)
new_graph_docs = transformer.convert_to_graph_documents(new_doc)

# 查看提取结果
for doc in new_graph_docs:
    print(f"节点: {[n.id for n in doc.nodes]}")
    for rel in doc.relationships:
        print(f"关系: {rel.source.id} -[{rel.type}]-> {rel.target.id}")

In [ ]:
# 增量写入（不会覆盖已有数据）
graph.add_graph_documents(new_graph_docs, baseEntityLabel=True, include_source=True)
print("\u2713 新数据已增量写入")

In [ ]:
# 验证新数据
result = graph.query(
    "MATCH (n) WHERE n.name CONTAINS '小米' OR n.id CONTAINS '小米' "
    "RETURN n.name AS name, n.id AS id, labels(n) AS labels"
)
for r in result:
    print(f"  {r}")

### 刚才发生了什么？

1. **LLMGraphTransformer** 用 LLM 从自然语言文本中提取实体和关系（第五章学过）
2. **graph.add_graph_documents()** 把提取的图数据写入 Neo4j
   - `baseEntityLabel=True`：给所有节点加一个 `__Entity__` 基础标签，方便统一查询
   - `include_source=True`：保存原始文档的引用，方便溯源
3. 增量写入使用的是 `MERGE` 语义——同名节点不会重复创建

### 增量更新 vs 全量重建

| | 增量更新 | 全量重建 |
|---|---|---|
| **速度** | 快，只处理新数据 | 慢，需要清空后重建 |
| **数据一致性** | 可能有冲突（新旧属性不一致） | 完全一致 |
| **适合场景** | 日常更新、新闻、动态数据 | 数据源变化大、需要清洗 |

> **生产建议**：日常用增量更新；定期（如每周）做一次全量重建来修正累积误差。

In [ ]:
# 用 Agent 验证：新数据已经可以被查询
response = agent.invoke({
    "messages": [{"role": "user", "content": "小米是谁创立的？在哪个城市？"}]
})
print(response["messages"][-1].content)

增量写入后，Agent 无需任何改动，就能立刻查到新数据——这就是工具化的好处：**数据更新和查询逻辑完全解耦**。

---

## 6. 性能优化

当知识图谱从几百个节点增长到几百万时，查询性能就成了关键问题。

In [ ]:
# 创建索引：加速按 name 属性查找
graph.query("CREATE INDEX company_name IF NOT EXISTS FOR (c:Company) ON (c.name)")
graph.query("CREATE INDEX person_name IF NOT EXISTS FOR (p:Person) ON (p.name)")
graph.query("CREATE INDEX city_name IF NOT EXISTS FOR (c:City) ON (c.name)")
graph.query("CREATE INDEX product_name IF NOT EXISTS FOR (p:Product) ON (p.name)")
print("\u2713 索引创建完成")

In [ ]:
# 查看所有索引
result = graph.query("SHOW INDEXES YIELD name, labelsOrTypes, properties, type")
for r in result:
    print(f"  {r['name']}: {r['labelsOrTypes']} -> {r['properties']} ({r['type']})")

In [ ]:
# EXPLAIN：查看查询计划（不实际执行）
result = graph.query(
    "EXPLAIN MATCH (c:Company)-[:LOCATED_IN]->(city:City) RETURN c.name, city.name"
)
print("查询计划:")
print(result)

In [ ]:
# PROFILE：实际执行并统计（会真正运行查询）
result = graph.query(
    "PROFILE MATCH (c:Company)-[:LOCATED_IN]->(city:City) RETURN c.name, city.name"
)
print("执行统计:")
print(result)

### 刚才发生了什么？

1. **创建索引**：`CREATE INDEX ... FOR (n:Label) ON (n.property)` 对指定标签的属性建索引
   - 有索引：Neo4j 直接跳到目标节点（类似字典查词）
   - 无索引：Neo4j 扫描该标签下所有节点（类似翻遍整本书）
2. **EXPLAIN**：输出查询计划但不执行，用于分析查询路径是否合理
3. **PROFILE**：执行查询并输出每一步的实际行数和耗时，用于定位性能瓶颈

### 性能优化策略汇总

| 优化策略 | 说明 | 示例 |
|---------|------|------|
| 创建索引 | 对常用查询属性建索引 | `CREATE INDEX ... ON (c.name)` |
| 使用 MERGE 而非 CREATE | 防止重复数据 | `MERGE (n:Company {name: $name})` |
| 参数化查询 | 使用 `$param` 而非字符串拼接（安全 + 缓存） | `WHERE n.name = $name` |
| LIMIT 结果集 | 避免返回过大数据集 | `RETURN ... LIMIT 100` |
| EXPLAIN / PROFILE | 查看查询计划和执行统计 | 定位慢查询 |
| 只读用户 | 生产环境使用只读数据库用户 | 防止误操作 |
| 连接池配置 | 合理设置 `max_connection_pool_size` | 避免连接耗尽 |

### 常见问题

- **索引没生效**：`CREATE INDEX` 是异步的，创建后需要等几秒才能生效。用 `SHOW INDEXES` 查看状态，确认 `state` 为 `ONLINE`。
- **EXPLAIN 和 PROFILE 的区别**：EXPLAIN 只分析不执行（零开销），PROFILE 会实际执行（可能很慢）。调试时先用 EXPLAIN，确认计划合理后再用 PROFILE。
- **参数化查询为什么更快**：Neo4j 会缓存查询计划。`WHERE n.name = $name` 的计划可以被复用，而 `WHERE n.name = '华为'` 和 `WHERE n.name = '腾讯'` 是两个不同的计划。
- **字符串拼接的安全风险**：`f"WHERE n.name = '{user_input}'"` 容易被 Cypher 注入攻击，务必使用参数化查询。

---

## 7. 学习路线回顾

回顾整个 Neo4j 教程的知识脉络，以及它和 LangChain / LangGraph 教程的关联。

| 章节 | 核心知识 | 与其他教程的关联 |
|------|---------|------------------|
| 01 | 图数据库基础、Docker 搭建 | — |
| 02 | Cypher 查询语言 | 对比 SQL |
| 03 | 数据建模、批量导入 | 对比 LangChain Document 加载 |
| 04 | LangChain Neo4j 集成 | LangChain ch03 LCEL、ch05 RAG |
| 05 | LLM 自动构建知识图谱 | LangChain ch02 Prompt Templates |
| 06 | GraphRAG 混合检索 | LangChain ch05 RAG |
| 07 | 图 Agent + 生产实践 | LangGraph ch04 工具 Agent |

### 完整知识链路

```
LangChain（基础框架）
  ├── Prompt、Chain、Memory、RAG、Agent
  └── 提供 @tool、Neo4jGraph 等基础组件
         ↓
LangGraph（复杂编排）
  ├── StateGraph、条件分支、工具 Agent
  └── 提供 create_react_agent、工作流编排能力
         ↓
Neo4j（知识图谱）
  ├── 图数据库、Cypher、GraphRAG
  └── 为 LLM 提供结构化的知识推理能力
```

三者的关系：
- **LangChain** 提供基础组件（模型调用、工具定义、文档处理）
- **LangGraph** 提供编排能力（状态管理、条件路由、Agent 循环）
- **Neo4j** 提供知识存储（实体关系、结构化推理、图检索）

它们各自解决不同层次的问题，组合起来才能构建真正的智能应用。

---

## 总结

本章学习了：

- \u2705 `@tool` 封装图查询为 Agent 工具（`query_graph`、`search_entity`、`get_graph_schema`）
- \u2705 `create_react_agent` 构建知识图谱 Agent，让 LLM 自主决策查询策略
- \u2705 LangGraph 工作流实现智能路由（图检索 / 直接回答）
- \u2705 增量更新知识图谱（`LLMGraphTransformer` + `add_graph_documents`）
- \u2705 性能优化策略（索引、EXPLAIN/PROFILE、参数化查询）

至此，Neo4j 知识图谱教程全部完成。你已经掌握了从图数据库基础到 GraphRAG 生产部署的完整知识链路。

## 进阶方向

- **图算法（GDS）**：社区检测、PageRank、图嵌入
- **多模态知识图谱**：结合图片、表格等非文本数据
- **企业级部署**：Neo4j 集群、备份恢复、监控告警
- **与 LlamaIndex 结合**：用 LlamaIndex 的 PropertyGraphIndex 构建更丰富的图索引